# 09 Comparison

Pulls together the matrix imaging results (05 noise-free, 06 noise, 07 model error) and the hypocentre results (08)
into one table per question. Node designs are judged on matrix imaging; the Trillium layout on earthquake locations.

05, 06 and 07 share one forward model, imaging step and score (`mi_common.py`), so their numbers can be compared directly.

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Results from 05-08 ---
mi = pd.read_csv("../data/results_05_mi.csv")
noise = pd.read_csv("../data/results_06_noise.csv")
merr = pd.read_csv("../data/results_07_model_error.csv")
hypo = pd.read_csv("../data/results_08_hypo.csv")

In [ ]:
REF = "1d"                             # focusing model treated as realistic for model error (07)
NSR = 1.0                              # noise level treated as realistic (one of the values scanned in 06)
DESIGNS = ["phased (adaptive)", "grid", "wide grid", "spiral"]

In [ ]:
# noise-free (05, focused with the true 3D model)
a = mi.groupby("design").coverage
# with noise (06), at the chosen level
b = noise[noise.nsr == NSR].groupby("design").coverage
# model error (07): worst over variants and scenarios
c = merr[merr.ref == REF].groupby("design").coverage

table = pd.DataFrame({
    "coverage, worst":            a.min(),
    "coverage, mean":             a.mean(),
    f"with noise {NSR}, worst":   b.min(),
    "model error, worst":         c.min(),
}).reindex(DESIGNS).round(2)

# phase 1: how far the BDT high picked from the phase 1 image was from the true one (adaptive design only)
ad = "phased (adaptive)"
p1 = {"noise-free": mi[mi.design == ad].p1_error_km,
      f"noise {NSR}": noise[(noise.design == ad) & (noise.nsr == NSR)].p1_error_km,
      "model error": merr[(merr.design == ad) & (merr.ref == REF)].p1_error_km}
table.loc[ad, "phase 1 error, worst (km)"] = round(max(s.max() for s in p1.values()), 1)

print(f"Node designs (model error focused with {REF})")
print(table)

In [ ]:
cols = ["coverage, worst", f"with noise {NSR}, worst", "model error, worst"]
labels = ["Ideal", f"Noise ({NSR})", "Wrong model"]

with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(7, 3.8))
    w = 0.8 / len(cols)
    for i, (col, lab) in enumerate(zip(cols, labels)):
        ax.bar(np.arange(len(DESIGNS)) + (i - 1) * w, table[col], w, label=lab)
    ax.set(xticks=range(len(DESIGNS)), xticklabels=[d.replace(' (', chr(10) + '(') for d in DESIGNS], ylim=(0, 1),
           ylabel="Worst-case coverage of BDT top", title=f"Node designs (model error focused with {REF})")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(6, 3.8))
    for d in DESIGNS:
        s = noise[noise.design == d].groupby("nsr").coverage.min()
        ax.plot(s.index, s.values, "o-", label=d)
    ax.axvline(NSR, color="grey", ls=":", lw=1)
    ax.set(xscale="log", ylim=(0, 1), xlabel="Noise / signal (30 days, reference)",
           ylabel="Worst-case coverage", title="Does the ranking depend on the noise level?")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
hyp = hypo.groupby(["network", "source"]).agg(
    located=("ez", lambda s: s.notna().mean()),
    horizontal_error_km=("eh", "median"),
    depth_error_km=("ez", lambda s: s.abs().median()),
    depth_bias_km=("ez", "median")).round(2)
print("Permanent network, earthquake locations")
print(hyp)